# 📚 Section 4: Indexing & Slicing
> Indexing & Slicing — Pulling out single elements vs. ranges, in 1D and 2D, and the View-vs-Copy trap that catches almost everyone once

---
# 🎯 Learning Objective
Today I want to learn:
- [x] The difference between indexing (single element) and slicing (a range), in both 1D and 2D
- [x] How to read and write `arr[start:stop:step]` and `arr[row, col]` / `arr[rows, cols]`
- [x] Why NumPy slices are Views (not copies) and when you must call `.copy()` instead

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

Indexing pulls out a single element (`arr[3]`); slicing pulls out a range (`arr[1:4]`). Both extend naturally to 2D with `arr[row, col]` for a single cell and `arr[rows, cols]` for a sub-matrix. The one thing that trips up almost everyone coming from Python lists: NumPy slicing returns a *View* — it shares memory with the original array — while Python list slicing always copies.

인덱싱은 단일 원소(`arr[3]`)를 꺼내고, 슬라이싱은 범위(`arr[1:4]`)를 꺼냅니다. 둘 다 2D로 자연스럽게 확장되어, 단일 셀은 `arr[row, col]`, 부분 행렬은 `arr[rows, cols]`로 꺼낼 수 있습니다. Python list에 익숙한 사람 대부분이 걸려 넘어지는 지점 하나: NumPy 슬라이싱은 *View*를 반환합니다 — 즉 원본과 메모리를 공유합니다 — 반면 Python list 슬라이싱은 항상 복사됩니다.

### What returns a View, and what returns a Copy? / 무엇이 View이고, 무엇이 Copy인가?

| Operation / 연산 | Returns / 반환 | Modifying it changes the original? / 원본에 영향? |
|---|---|---|
| `arr[a:b]` — slicing | View | Yes — shares memory (메모리 공유) |
| `arr[a:b, c:d]` — 2D slicing | View | Yes — shares memory (메모리 공유) |
| `arr[bool_mask]` — Boolean Indexing (Sec. 5) | Copy | No — independent (독립적) |
| `arr[[i, j, k]]` — Fancy Indexing (Sec. 5) | Copy | No — independent (독립적) |
| `arr[a:b].copy()` — explicit copy | Copy | No — independent (독립적) |

## Why do we use it?
*(When is it useful?)*

Because almost every real analysis starts by pulling out a piece of the data — this month, this region, the last N rows — and knowing whether that piece is a live window into the original or an independent snapshot decides whether your cleanup step accidentally corrupts your source data.

거의 모든 실제 분석이 데이터의 일부 — 이번 달, 이 지역, 최근 N개 행 — 를 꺼내는 것에서 시작하기 때문입니다. 꺼낸 조각이 원본을 들여다보는 창인지, 독립된 스냅샷인지 아는 것이 데이터 정제 단계에서 원본 데이터를 실수로 훼손하는지 여부를 결정합니다.

## When is it used in Business Analytics?
*(Real-world use case)*

- Grabbing the first/last day's value, or the most recent value in a running series.  
첫째 날/마지막 날 값이나, 누적 시계열에서 가장 최근 값을 꺼낼 때.
- Splitting a year of data into H1/H2, or pulling the most recent N days for a rolling metric.  
1년치 데이터를 상반기/하반기로 나누거나, 롤링 지표를 위해 최근 N일을 꺼낼 때.
- Looking up one region's one quarter in a pivot-style 2D array.  
피벗 형태의 2D 배열에서 특정 지역의 특정 분기를 조회할 때.
- Cleaning outliers or missing values WITHOUT corrupting the original raw array.  
  원본 raw 배열을 훼손하지 않고 이상치·결측치를 정제할 때.

---
# 📝 Syntax

## Basic Syntax

In [1]:
import numpy as np

arr = np.array([10, 20, 30, 40, 50])

# 1D indexing — single element
print(arr[0])     # 10 -> first
print(arr[-1])    # 50 -> last

# 1D slicing — a range, stop excluded
print(arr[1:4])   # [20 30 40]

# 2D indexing/slicing
grid = np.array([[1, 2, 3], [4, 5, 6]])
print(grid[0, 1])    # 2 -> row 0, col 1
print(grid[:, 1])    # [2 5] -> every row, col 1

10
50
[20 30 40]
2
[2 5]


## Common Variations

In [2]:
import numpy as np

arr = np.array([10, 20, 30, 40, 50])

# step and reverse
print(arr[::2])    # [10 30 50] -> every other element
print(arr[::-1])   # [50 40 30 20 10] -> reversed

# omitting start/stop
print(arr[:3])     # first 3
print(arr[2:])     # from index 2 to the end

# View vs Copy in one glance
view = arr[1:3]
copy = arr[1:3].copy()
view[0] = 999
print(arr)               # changed! -> slice is a View
print(copy.base is arr)  # False -> copy is independent

[10 30 50]
[50 40 30 20 10]
[10 20 30]
[30 40 50]
[ 10 999  30  40  50]
False


---
# 🧪 Small Examples

## Example 1: 1D Indexing — Daily Active Users, Day-over-Day
### 4-1. 1D 인덱싱 (양수 / 음수)

In [3]:
import numpy as np

# Business: 30 days of Daily Active Users (DAU)
dau = np.array([
    4500, 4800, 5100, 4900, 5300, 5600, 5200,
    4700, 4900, 5000, 5100, 5400, 5700, 5300,
    4800, 5000, 5200, 5500, 5800, 5400, 5100,
    4900, 5100, 5300, 5600, 5900, 5500, 5200,
    5000, 5100
])

print(f"First day of the month: {dau[0]:,}")
print(f"Last day of the month: {dau[-1]:,}")
print(f"Day-over-day change: {dau[-1] - dau[-2]:+,}")

# Growth rate: last day vs first day
growth_rate = (dau[-1] - dau[0]) / dau[0] * 100
print(f"Monthly DAU growth: {growth_rate:+.1f}%")

First day of the month: 4,500
Last day of the month: 5,100
Day-over-day change: +100
Monthly DAU growth: +13.3%


## Example 2: 1D Slicing — Splitting a Year into Halves
### 4-2. 1D 슬라이싱 (start:stop:step)

In [4]:
import numpy as np

# Business: split annual sales into H1/H2 and compare
annual_sales = np.array([1200, 1350, 1100, 1400, 1600, 1500,
                          1300, 1250, 1450, 1700, 1800, 2000])

h1 = annual_sales[:6]     # H1: Jan-Jun
h2 = annual_sales[6:]     # H2: Jul-Dec

print(f"H1 total: {h1.sum():,}")
print(f"H2 total: {h2.sum():,}")
print(f"H2 growth vs H1: {(h2.sum() - h1.sum()) / h1.sum() * 100:.1f}%")

# Business: recent 3-month rolling average
last_3 = annual_sales[-3:]
print(f"\nRecent 3-month average: {last_3.mean():.0f}")
print(f"Recent 3-month values: {last_3}")

H1 total: 8,150
H2 total: 9,500
H2 growth vs H1: 16.6%

Recent 3-month average: 1833
Recent 3-month values: [1700 1800 2000]


## Example 3: 2D Indexing — Looking Up One Region, One Quarter
### 4-3. 2D 인덱싱 — [row, col]

In [5]:
import numpy as np

# Business: 3 regions x 4 quarters sales table
region_sales = np.array([
    [1200, 1400, 1300, 1700],   # Seoul
    [800, 950, 870, 1100],      # Busan
    [600, 720, 680, 900]        # Daegu
])

regions = ["Seoul", "Busan", "Daegu"]
quarters = ["Q1", "Q2", "Q3", "Q4"]

# Look up one region's one quarter
target_region, target_q = 0, 3   # Seoul, Q4
print(f"{regions[target_region]} {quarters[target_q]}: "
      f"{region_sales[target_region, target_q]:,}")

# Loop through all regions' Q4 performance
print("\nQ4 by region:")
for i, region in enumerate(regions):
    print(f"  {region}: {region_sales[i, -1]:,}")

Seoul Q4: 1,700

Q4 by region:
  Seoul: 1,700
  Busan: 1,100
  Daegu: 900


## Example 4: 2D Slicing — Marketing Channel Conversion by Quarter
### 4-4. 2D 슬라이싱 — [row범위, col범위]

In [6]:
import numpy as np

# Business: 6 months x 5 channels conversion rate (%)
conversion = np.array([
    [3.2, 4.1, 2.8, 5.0, 2.5],   # Jan
    [3.4, 4.3, 3.0, 5.2, 2.6],   # Feb
    [3.1, 4.0, 2.7, 4.9, 2.4],   # Mar
    [3.6, 4.5, 3.2, 5.4, 2.8],   # Apr
    [3.8, 4.7, 3.4, 5.6, 3.0],   # May
    [3.5, 4.4, 3.1, 5.3, 2.7],   # Jun
])
channels = ["organic", "paid_ads", "sns", "email", "direct"]

# Q1 (Jan-Mar), just paid_ads (col 1) and email (col 3)
q1_paid_email = conversion[0:3, [1, 3]]
print("Q1 paid_ads & email conversion:\n", q1_paid_email)
print(f"Q1 paid_ads average: {q1_paid_email[:, 0].mean():.2f}%")
print(f"Q1 email average: {q1_paid_email[:, 1].mean():.2f}%")

Q1 paid_ads & email conversion:
 [[4.1 5. ]
 [4.3 5.2]
 [4.  4.9]]
Q1 paid_ads average: 4.13%
Q1 email average: 5.03%


## Example 5: View vs Copy — Cleaning Outliers Without Corrupting Raw Data
### 4-5. View vs Copy — 슬라이스는 View다!

In [7]:
import numpy as np

raw_sales = np.array([1200, 1350, 99999, 1400, 1600, -500, 1300])
#                                ^outlier          ^bad value
print("Raw data:", raw_sales)

# X Wrong way — modifying a View also changes the original
bad_slice = raw_sales[:]   # this is still a View
bad_slice[2] = 0
print("Original after modifying the View:", raw_sales)   # corrupted!

# Correct way — copy() first, then clean
raw_sales2 = np.array([1200, 1350, 99999, 1400, 1600, -500, 1300])
clean_sales = raw_sales2.astype(float)   # independent copy AND float, so it can hold np.nan too

clean_sales[clean_sales < 0] = 0
clean_sales[clean_sales > 5000] = np.nan

print("\nCleaned data:", clean_sales)
print("Original preserved:", raw_sales2)   # untouched

Raw data: [ 1200  1350 99999  1400  1600  -500  1300]
Original after modifying the View: [1200 1350    0 1400 1600 -500 1300]

Cleaned data: [1200. 1350.   nan 1400. 1600.    0. 1300.]
Original preserved: [ 1200  1350 99999  1400  1600  -500  1300]


## Example 6: Practice — One Mini-Exercise per Subtopic
### 연습 문제 (Practice Problems, 4-1 ~ 4-5)

**Task / 과제:** This section's original material has one practice problem per subtopic — solve all five below.
이번 섹션은 하위주제마다 연습 문제가 하나씩 있습니다 — 아래 5개를 모두 풀어보세요.

1. **(4-1)** From `weekly = [850, 920, 780, 1100, 1350]` (Mon-Fri), get Wednesday and Friday, print their difference.  
`weekly = [850, 920, 780, 1100, 1350]`(월~금)에서 수요일과 금요일을 꺼내 차이를 출력하세요.
2. **(4-2)** From a 30-day visitors array (`np.arange(100, 130)`), get (a) days 1-10, (b) the last 7 days, (c) every 5th day.  
30일치 방문자 배열(`np.arange(100, 130)`)에서 (a) 1~10일, (b) 마지막 7일, (c) 5일 간격 데이터를 꺼내세요.
3. **(4-3)** From a 3-employee × 4-quarter performance table, get one employee's Q3, all employees' Q4, and one employee's full year.  
3명 × 4분기 성과 테이블에서 한 직원의 Q3, 전 직원의 Q4, 한 직원의 전체 분기를 꺼내세요.
4. **(4-4)** From a 5-product × 4-week sales table, get the top 3 products' weeks 2-3.    
5개 상품 × 4주 판매량 테이블에서 상위 3개 상품의 2~3주차 데이터를 꺼내세요.
5. **(4-5)** Make a slice (View), modify it and confirm the original changes; then make a `.copy()`, modify it, and confirm the original is preserved.   
슬라이스(View)를 만들어 수정한 뒤 원본이 바뀌는 것을 확인하고, `.copy()`로 만든 복사본을 수정한 뒤 원본이 보존되는 것을 확인하세요.

Fill in each `________` below, then run the cell.
아래 각 `________`를 채운 후 셀을 실행하세요.

In [14]:
# ✏️ Practice — replace each ________ line below, then run this cell.
# ✏️ 연습 문제 — 아래 각 ________ 줄을 채운 후 셀을 실행하세요.

import numpy as np

print("--- 4-1: 1D indexing ---")
# TODO: weekly = [850,920,780,1100,1350] -> get Wed (idx 2) and Fri (last), print the difference
weekly = [850,920,780,1100,1350]
print(weekly[2] - weekly[-1])

print("\n--- 4-2: 1D slicing ---")
# TODO: visitors = np.arange(100,130) -> print days 1-10, last 7 days, and every 5th day
visitors = np.arange(100,130)
print(visitors[:10])
print(visitors[-7:])
print(visitors[::5])

print("\n--- 4-3: 2D indexing ---")
# TODO: perf = [[45,52,48,61],[38,44,41,55],[50,58,54,70]] -> print emp[2]'s Q3, all Q4, emp[1]'s full year
perf = np.array([[45,52,48,61],[38,44,41,55],[50,58,54,70]])
print(perf[2, 2])   # emp[2]'s Q3
print(perf[:, 3])   # all employees' Q4
print(perf[1])      # emp[1]'s full year

print("\n--- 4-4: 2D slicing ---")
# TODO: weekly_sales (5 products x 4 weeks) -> print top 3 products' weeks 2-3 (rows 0:3, cols 1:3)
weekly_sales = np.array([[1200, 1400, 1300, 1700],
                         [800, 950, 870, 1100],
                         [600, 720, 680, 900],
                         [1000, 1100, 1050, 1300],
                         [1100, 1250, 1450, 1700]])
print(weekly_sales[0:3, 1:3])


print("\n--- 4-5: View vs Copy ---")
# TODO: make a view of an array, modify it, print original (changed); then a .copy(), modify it, print original (unchanged)
arr = np.array([1,2,3,4,5])
view = arr[:]
view[0] = 99
print(arr)

copy = arr.copy()
copy[1] = 88
print(arr)
print("\n✅ Fill in each ________ above with real code, then re-run to see all 5 results.")
print("✅ 위 각 ________ 를 실제 코드로 채운 뒤 다시 실행하면 5개 결과를 모두 볼 수 있습니다.")

--- 4-1: 1D indexing ---
-570

--- 4-2: 1D slicing ---
[100 101 102 103 104 105 106 107 108 109]
[123 124 125 126 127 128 129]
[100 105 110 115 120 125]

--- 4-3: 2D indexing ---
54
[61 55 70]
[38 44 41 55]

--- 4-4: 2D slicing ---
[[1400 1300]
 [ 950  870]
 [ 720  680]]

--- 4-5: View vs Copy ---
[99  2  3  4  5]
[99  2  3  4  5]

✅ Fill in each ________ above with real code, then re-run to see all 5 results.
✅ 위 각 ________ 를 실제 코드로 채운 뒤 다시 실행하면 5개 결과를 모두 볼 수 있습니다.


---
# ⚠️ Common Mistakes

**Mistake 1 — Indexing past the last valid position.**
If an array has 7 elements, valid indices are `0` through `6` — `sales[7]` raises an `IndexError`, not a friendly empty result.     
배열에 원소가 7개면 유효한 인덱스는 `0`부터 `6`까지입니다 — `sales[7]`은 빈 결과가 아니라 `IndexError`를 발생시킵니다.  
✅ **Fix:** Remember the last valid positive index is always `len(arr) - 1`.    
✅ **해결법:** 마지막 유효한 양수 인덱스는 항상 `len(arr) - 1`이라는 걸 기억하세요.

**Mistake 2 — Reaching for `-0` to mean "the last element."**
`-0` is just `0` (the first element) — negative indexing starts at `-1` for the last element, not `-0`.     
`-0`은 그냥 `0`(첫 번째 원소)입니다 — 음수 인덱싱은 마지막 원소부터 `-1`로 시작하며, `-0`이 아닙니다.   
✅ **Fix:** The last element is always `arr[-1]`.   
✅ **해결법:** 마지막 원소는 항상 `arr[-1]`입니다.

**Mistake 3 — Forgetting that `stop` is excluded in a slice.**
`monthly[1:3]` returns exactly 2 elements (indices 1 and 2) — index 3 is NOT included. The element count of any slice is `stop - start`.    
`monthly[1:3]`은 정확히 2개(인덱스 1, 2)를 반환합니다 — 인덱스 3은 포함되지 않습니다. 슬라이스의 원소 개수는 항상 `stop - start`입니다. 
✅ **Fix:** When you want to include a specific end index, add 1 to `stop`. 
✅ **해결법:** 특정 끝 인덱스를 포함하고 싶다면 `stop`에 1을 더하세요.

**Mistake 4 — Mismatching array index with real-world position.**
`monthly[::2]` selects even-numbered ARRAY INDICES (0, 2, 4...), which are actually the 1st, 3rd, 5th... calendar months, because arrays are 0-indexed but months are 1-indexed.    
`monthly[::2]`는 짝수 배열 인덱스(0, 2, 4...)를 선택하는데, 배열은 0부터 시작하고 달(month)은 1부터 시작하므로 실제로는 1월, 3월, 5월... 이 선택됩니다.     
✅ **Fix:** When mapping indices to real-world labels (day 1, month 1...), always double-check the off-by-one — write out the first couple of indices by hand if unsure.    
✅ **해결법:** 인덱스를 실제 라벨(1일, 1월...)에 매핑할 때는 항상 off-by-one을 다시 확인하세요 — 확신이 안 서면 처음 몇 개 인덱스를 직접 손으로 적어보세요.

**Mistake 5 — Writing `sales[col, row]` instead of `sales[row, col]`.**
2D access is always row first, column second. `sales[1][1]` (chained indexing) happens to work too, but `sales[1, 1]` is the NumPy-idiomatic form and performs better.      
2D 접근은 항상 행이 먼저, 열이 나중입니다. `sales[1][1]`(체이닝 인덱싱)도 동작은 하지만, `sales[1, 1]`이 NumPy 관용 표현이며 성능도 더 좋습니다.    
✅ **Fix:** Say it out loud as "row, column" every time until it's automatic; use the comma form `arr[r, c]`, not chained brackets.     
✅ **해결법:** 자동이 될 때까지 매번 "행, 열"이라고 소리 내어 말해보세요. 체이닝 대괄호가 아니라 쉼표 형태 `arr[r, c]`를 사용하세요.

**Mistake 6 — Chaining `[0:2][3:6]` instead of combining into `[0:2, 3:6]`.**
`visitors[0:2][3:6]` first slices rows 0-1, then tries to slice ROWS 3-6 of that 2-row result — which is empty or wrong, not "rows 0-1, columns 3-6."   
`visitors[0:2][3:6]`는 먼저 행 0~1을 슬라이싱한 뒤, 그 2개 행짜리 결과에서 다시 행 3~6을 슬라이싱하려 합니다 — "행 0~1, 열 3~6"이 아니라 비어있거나 잘못된 결과가 나옵니다.     
✅ **Fix:** Always combine row and column selection with one comma inside one set of brackets: `visitors[0:2, 3:6]`.    
✅ **해결법:** 행과 열 선택은 항상 하나의 대괄호 안에 쉼표로 합치세요: `visitors[0:2, 3:6]`.

**Mistake 7 — Assuming `arr[:]` gives you an independent copy.**
`arr[:]` LOOKS like "the whole thing, copied," but it's still just a slice — still a View that shares memory with `arr`.    
`arr[:]`는 "전체를 복사한 것"처럼 보이지만, 여전히 슬라이스일 뿐이며 `arr`와 메모리를 공유하는 View입니다.  
✅ **Fix:** For a real independent copy, always call `.copy()` explicitly: `arr[:].copy()` or simply `arr.copy()`.  
✅ **해결법:** 진짜 독립적인 복사본이 필요하면 항상 `.copy()`를 명시적으로 호출하세요: `arr[:].copy()` 또는 그냥 `arr.copy()`.

**Mistake 8 — Not realizing Boolean and Fancy Indexing behave differently from slicing.**
Unlike a slice, `arr[arr > 0]` (Boolean Indexing) and `arr[[0, 2, 4]]` (Fancy Indexing) ALWAYS return a Copy — the opposite default from slicing. These are covered in depth in Section 5.  
슬라이스와 달리, `arr[arr > 0]`(Boolean Indexing)과 `arr[[0, 2, 4]]`(Fancy Indexing)는 항상 Copy를 반환합니다 — 슬라이싱과 반대되는 기본 동작입니다. 이 내용은 섹션 5에서 자세히 다룹니다.  
✅ **Fix:** Don't assume every kind of indexing shares the same View/Copy behavior — the selection METHOD (position-based vs. condition-based) determines it.   
✅ **해결법:** 모든 인덱싱 방식이 같은 View/Copy 동작을 한다고 가정하지 마세요 — 선택 방식(위치 기반 vs. 조건 기반)이 이를 결정합니다.

---
# 💡 Tips
Useful tips or shortcuts

- Print `something.base is original_array` right after any slice if you're unsure whether you're holding a View or a Copy — `True` means View (shared memory), `None` means independent.  
  슬라이스가 View인지 Copy인지 확실치 않다면 바로 `something.base is original_array`를 출력해보세요 — `True`면 View(메모리 공유), `None`이면 독립적입니다.
- Before cleaning or transforming any array you don't want to lose, write `.copy()` as a reflex — it costs almost nothing and prevents an entire category of "why did my raw data change?!" bugs.   
  잃고 싶지 않은 배열을 정제·변환하기 전에는 반사적으로 `.copy()`를 쓰세요 — 비용은 거의 없고, "원본 데이터가 왜 바뀌었지?!" 유형의 버그 전체를 예방합니다.
- When 2D indexing feels confusing, say it out loud: "row, then column" — `arr[row, col]`, always in that order.  
  2D 인덱싱이 헷갈릴 땐 소리 내어 말해보세요: "행, 그다음 열" — `arr[row, col]`, 항상 이 순서입니다.

---
# 🔗 Related Concepts

```
Section 3 — Array Attributes
   (.shape, .ndim, .dtype, .size)
        ↓
🔵 Section 4 — Indexing & Slicing   ← you are here
   (single elements & ranges, 1D & 2D, View vs Copy)
        ↓
Section 5 — Boolean & Fancy Indexing
   (selecting by CONDITION instead of position)
        ↓
Sections 6-9 — manipulation,
   vectorization, aggregation, random/sorting
        ↓
Section 10 — Pandas + NumPy integration
        ↓
Sections 11-12 — Business KPIs, missing values
        ↓
Pandas → SQL → Tableau
```

*How is today's topic connected to other concepts?*

Section 3 told you HOW MANY rows and columns you have; this section is how you actually reach into them by position. Section 5 immediately builds on this by selecting elements based on a CONDITION instead of a fixed position — and its results always behave like the `.copy()` you learned about here, never like a View.

섹션 3이 몇 개의 행과 열이 있는지 알려줬다면, 이번 섹션은 실제로 위치로 그것들에 접근하는 방법입니다. 섹션 5는 고정된 위치가 아닌 조건(condition)으로 원소를 선택하는 방식으로 곧바로 이어지는데, 그 결과는 항상 여기서 배운 `.copy()`처럼 동작하며, View처럼 동작하지 않습니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
You're preparing a quarterly marketing report. You need to (1) pull the most recent week's sales using negative indexing, (2) split a year of sales into H1/H2, (3) look up one specific region-quarter cell, and (4) clean outliers from raw data WITHOUT corrupting the source array your teammates still need.       
분기별 마케팅 리포트를 준비하는 상황입니다. 필요한 것: (1) 음수 인덱싱으로 최근 주 매출 꺼내기, (2) 1년 매출을 상반기/하반기로 나누기, (3) 특정 지역-분기 셀 조회, (4) 동료들이 여전히 필요로 하는 원본 배열을 훼손하지 않고 이상치 정제하기.

**To-do / 할 일:**
- [x] Use negative indexing to grab the most recent week's data.  
      음수 인덱싱으로 최근 주 데이터를 꺼냅니다.
- [x] Slice the year into H1 and H2, and compare totals.    
      1년치를 상반기·하반기로 슬라이싱하고 합계를 비교합니다.
- [x] Use 2D indexing to look up one region's one quarter.  
      2D 인덱싱으로 특정 지역의 특정 분기를 조회합니다.
- [x] Use `.copy()` before cleaning outliers, and confirm the original is untouched.      
      이상치를 정제하기 전 `.copy()`를 사용하고, 원본이 그대로인지 확인합니다.

In [9]:
import numpy as np

# 1. Weekly sales for the year — grab the most recent week
weekly_sales = np.array([820, 910, 875, 940, 1020, 980, 1100, 1050,
                          990, 1150, 1080, 1200])
print("Most recent week's sales:", weekly_sales[-1])

# 2. H1 vs H2 comparison
h1, h2 = weekly_sales[:6], weekly_sales[6:]
print(f"H1 total: {h1.sum()}, H2 total: {h2.sum()}")

# 3. Region x quarter lookup
region_sales = np.array([
    [1200, 1400, 1300, 1700],
    [800, 950, 870, 1100],
    [600, 720, 680, 900]
])
print("Busan Q3:", region_sales[1, 2])

# 4. Safe outlier cleanup — copy first!
raw = np.array([1200, 1350, 99999, 1400, -500, 1600])
clean = raw.astype(float)   # independent copy AND float, so it can hold np.nan too
clean[clean < 0] = 0
clean[clean > 5000] = np.nan

print("\nCleaned:", clean)
print("Original still intact:", raw)

Most recent week's sales: 1200
H1 total: 5545, H2 total: 6570
Busan Q3: 870

Cleaned: [1200. 1350.   nan 1400.    0. 1600.]
Original still intact: [ 1200  1350 99999  1400  -500  1600]


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

Indexing pulls a single element (`arr[i]` or `arr[row, col]`); slicing pulls a range (`arr[a:b]` or `arr[a:b, c:d]`), with `stop` always excluded and a `step` you can use to skip elements or reverse. Negative indices count from the end (`-1` is last), and 2D access is always row-then-column. The concept worth internalizing above all others: NumPy slices are Views that share memory with the original, so modifying a slice modifies the source array too — call `.copy()` explicitly whenever you need an independent snapshot, which matters most right before any data-cleaning step.

인덱싱은 단일 원소(`arr[i]` 또는 `arr[row, col]`)를 꺼내고, 슬라이싱은 범위(`arr[a:b]` 또는 `arr[a:b, c:d]`)를 꺼내며, `stop`은 항상 미포함이고 `step`으로 건너뛰거나 역순으로 만들 수 있습니다. 음수 인덱스는 끝에서부터 세며(`-1`이 마지막), 2D 접근은 항상 행 다음 열 순서입니다. 무엇보다 꼭 체화해야 할 개념: NumPy 슬라이스는 원본과 메모리를 공유하는 View이므로, 슬라이스를 수정하면 원본도 함께 바뀝니다 — 독립적인 스냅샷이 필요할 때는 항상 `.copy()`를 명시적으로 호출해야 하며, 이는 데이터 정제 작업 직전에 가장 중요합니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Indexing grabs one element and slicing grabs a range — in 1D or 2D — but the detail that matters most is that a NumPy slice is a View sharing memory with the original, so you need `.copy()` whenever you want an independent snapshot.

> 인덱싱은 원소 하나를, 슬라이싱은 범위를 꺼내지만(1D든 2D든), 가장 중요한 디테일은 NumPy 슬라이스가 원본과 메모리를 공유하는 View라는 점이며, 독립적인 스냅샷이 필요할 때는 반드시 `.copy()`를 사용해야 합니다.

---
# ❓ Review Questions

**Q1.** What's the difference between `arr[3]` and `arr[3:4]` — both seem to "get the 4th element"?     
`arr[3]`과 `arr[3:4]`는 둘 다 "4번째 원소를 가져오는" 것처럼 보이는데, 차이가 무엇인가요?
arr[3] returns the actual element, while arr[3:4] returns a one-element array (a slice).  
arr[3]은 4번째 원소 자체를 반환하고, arr[3:4]는 4번째 원소를 포함한 1개의 원소짜리 배열을 반환합니다.

**Q2.** Why does `monthly[::2]` return the 1st, 3rd, 5th... elements instead of the 2nd, 4th, 6th?  
`monthly[::2]`는 왜 2번째, 4번째, 6번째가 아니라 1번째, 3번째, 5번째... 원소를 반환하나요?
Slicing uses zero-based indexing, so the indices are 0, 1, 2, 3.... A step of 2 means "take every 2 indices", starting from index 0.  
Python의 인덱스는 0부터 시작하기 때문에 0, 1, 2, 3...이 됩니다. step=2는 현재 위치에서 2칸씩 이동한다는 뜻이고, 기본 시작점은 0입니다.

**Q3.** In `arr[row, col]`, which comes first — row or column — and what happens if you swap them?  
`arr[row, col]`에서 행과 열 중 무엇이 먼저 오나요? 순서를 바꾸면 어떻게 되나요?
The row comes first, followed by the column. Swapping them accesses a different position in the array.  
arr[row, col]에서는 행(row)이 먼저, 그 다음에 **열(column)**이 옵니다. 순서를 바꾸면 다른 위치의 값을 가져옵니다.

**Q4.** If you slice an array and then modify the slice, what happens to the original array, and why?   
배열을 슬라이싱한 뒤 슬라이스를 수정하면 원본 배열은 어떻게 되며, 그 이유는 무엇인가요?
A NumPy slice usually returns a view, so modifying the slice also modifies the original array because they share the same underlying data.  
NumPy에서 slicing한 결과는 보통 View이기 때문에, 슬라이스와 원본이 같은 데이터를 공유합니다. 따라서 슬라이스를 수정하면 원본도 변경됩니다.

**Q5.** Name two ways to get an independent copy of a NumPy array instead of a View.    
NumPy 배열의 View 대신 독립적인 복사본을 얻는 방법 두 가지를 말해보세요.
You can use .copy() or np.copy() to create an independent copy.  
View가 아닌 독립적인 복사본을 만들려면 .copy() 또는 np.copy()를 사용할 수 있습니다.

---
*📅 Try answering these again in a few days.*